# Setup and Imports

In [ ]:
import importlib
import src.data_loader
import src.heuristic
import src.gnn_utils
import src.gnn_model

importlib.reload(src.data_loader)
importlib.reload(src.heuristic)
importlib.reload(src.gnn_utils)
importlib.reload(src.gnn_model)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings
import os

from src.data_loader import (
    load_wikispeedia_file,
    load_shortest_path_matrix,
    get_article_text,
    get_article_html,
    find_absolute_link_position_from_df
)
from src.feature_extractor import FeatureExtractor, EMBEDDING_DIM
from src.evaluation import calculate_mrr
from src.html_processor import HtmlProcessor
from pathlib import Path
from src.heuristic import (
    expand_paths_to_steps,
    compute_mrr_next_link
)
import random
from src.gnn_utils import *
from src.gnn_model import *
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

## Load .tsv and .txt files

In [ ]:
articles_df = load_wikispeedia_file(
    "articles.tsv",
    columns=["article_name"]
)
articles_df["article_id"] = articles_df["article_name"]

print("Articles:", articles_df.shape)
display(articles_df.head())

categories_df = load_wikispeedia_file(
    "categories.tsv",
    columns=["article", "category"]
)
print("Categories:", categories_df.shape, "\n")
display(categories_df.head())

links_df = load_wikispeedia_file(
    "links.tsv",
    columns=["source", "target"]
)
print("Links:", links_df.shape)
display(links_df.head())

paths_finished_df = load_wikispeedia_file(
    "paths_finished.tsv",
    columns=["hashedIpAddress", "timestamp", "durationInSec", "path", "rating"]
)
print("Finished paths:", paths_finished_df.shape)
display(paths_finished_df.head())

paths_unfinished_df = load_wikispeedia_file(
    "paths_unfinished.tsv",
    columns=["session_id", "timestamp", "session_time", "path", "rating", "type"]
)
print("Unfinished paths:", paths_unfinished_df.shape)
display(paths_unfinished_df.head())

dist_matrix = load_shortest_path_matrix()
print(dist_matrix.shape)

## Load plain article text & html text into articles dataframe

In [ ]:
articles_df["plaintext"] = articles_df["article_name"].apply(lambda x: get_article_text(x))

articles_df["plaintext_length"] = articles_df["plaintext"].apply(
    lambda x: len(x) if isinstance(x, str) else 0
)

articles_df["html_content"] = articles_df["article_name"].apply(
    lambda x: get_article_html(x)
)

### Load html text for 5 articles with errors manually 

In [ ]:
manual_fixes = {
    "Directdebit": "./data/wikispeedia_articles_html/wpcd/wp/d/Directdebit.htm"
    ,"Donation": "./data/wikispeedia_articles_html/wpcd/wp/d/Donation.htm"
    ,"Friend_Directdebit": "./data/wikispeedia_articles_html/wpcd/wp/f/Friend_Directdebit.htm"
    ,"Sponsorship_Directdebit": "./data/wikispeedia_articles_html/wpcd/wp/s/Sponsorship_Directdebit.htm"
    ,"Wowpurchase": "./data/wikispeedia_articles_html/wpcd/wp/w/Wowpurchase.htm"
}

for name, path in manual_fixes.items():
    file_path = Path(path)
    try:
        html_text = file_path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        html_text = file_path.read_text(encoding="latin-1")
    articles_df.loc[articles_df["article_name"] == name, "html_content"] = html_text
    print(f"Loaded {name}")

nan_html_content = articles_df[articles_df["html_content"].isna()]
print(f"Amount articles with missing html content: {len(nan_html_content)}")
display(nan_html_content)

### Add column with length of html text

In [ ]:
articles_df["html_length"] = articles_df["html_content"].apply(
    lambda x: len(x) if isinstance(x, str) else 0
)

display(articles_df.sort_values(by="html_length", ascending=True))

## Add two columns for indegree and outdegree of each article

In [ ]:
outdegree = links_df["source"].value_counts().rename("outdegree")
indegree = links_df["target"].value_counts().rename("indegree")

articles_df = articles_df.drop(columns=["outdegree", "indegree"], errors="ignore")
articles_df = (
    articles_df
    .merge(outdegree, how="left", left_on="article_name", right_index=True)
    .merge(indegree, how="left", left_on="article_name", right_index=True)
)

articles_df[["outdegree", "indegree"]] = articles_df[["outdegree", "indegree"]].fillna(0).astype(int)
isolated_df = articles_df[(articles_df["outdegree"] == 0) | (articles_df["indegree"] == 0)]

print(f"Amount articles with 0 indegree or outdegree: {len(isolated_df)}")
display(isolated_df)

## Add the position of the links in the html files to the links dataframe

In [ ]:
articles_lookup = dict(zip(articles_df["article_name"], articles_df["html_content"]))

links_df["link_position_absolute"] = links_df.apply(
    lambda row: find_absolute_link_position_from_df(row, articles_lookup)
    ,axis=1
)

links_df["html_length"] = links_df["source"].map(
    articles_df.set_index("article_name")["html_length"]
)

links_df["link_position_relative"] = (
    links_df["link_position_absolute"] / links_df["html_length"]
).where(links_df["html_length"] > 0)


In [ ]:
print("Sort absolute link position ascending")
display(links_df.sort_values(by="link_position_absolute", ascending=True))

print("Check for link position that couldn't be found")
nan_links_link_position = links_df[links_df["link_position_absolute"].isna()]
display(nan_links_link_position)

print("Sort relative link position ascending")
display(links_df.sort_values(by="link_position_relative"))

We have two links where the position cannot be determined -> We have to decide how to treat those entries

# Data loading and initial descriptive Analysis

In [ ]:
print("--- Descriptive Analysis ---")
print("--- About Articles and Links ---")

print(f"Total articles (nodes): {len(articles_df)}")
print(f"Total links (edges): {len(links_df)}")

paths_finished_df['path_length'] = paths_finished_df['path'].apply(lambda x: len(x.split(';')))
print(f"\n--- Path Length Statistics ---")
print(paths_finished_df['path_length'].describe())

In [ ]:
print("--- About finished paths ---")

plt.figure(figsize=(10, 6))
sns.histplot(paths_finished_df['path_length'], kde=False, bins=20)
plt.title('Distribution of Path Lengths')
plt.xlabel('Path Length (number of articles)')
plt.ylabel('Frequency')
plt.show()

# Pre-computation and Setup

In [ ]:
HTML_DIR = "./data/wikispeedia_articles_html/wpcd/wp"

if os.path.exists(HTML_DIR):
    html_proc = HtmlProcessor(HTML_DIR)
    link_position_map = html_proc.precompute_positions(articles_df['article_id'].tolist())
    print("Link positions extraction complete.")
else:
    print("HTML Directory not found. Link position feature will be filled with defaults (1.0).")
    link_position_map = {}

extractor = FeatureExtractor()

article_id_to_index = {
    article_id: index 
    for index, article_id in enumerate(articles_df['article_id'])
}

article_id_to_name = pd.Series(
    articles_df.article_name.values, 
    index=articles_df.article_id
).to_dict()

article_name_to_id = pd.Series(
    articles_df.article_id.values,
    index=articles_df.article_name
).to_dict()

links_map = links_df.groupby('source')['target'].apply(list).to_dict()
pagerank_map, out_degree_map = extractor.create_topology_maps(links_df)

embedding_map = {}
for article_id in tqdm(articles_df['article_id'], desc="Generating Embeddings"):
    article_name = article_id_to_name.get(article_id)
    if article_name:
        text = get_article_text(article_name)
        embedding_map[article_id] = extractor.get_text_embedding(text, strategy='first_para')
    else:
        embedding_map[article_id] = np.zeros(EMBEDDING_DIM)

print("Finished.")


Data Pipeline: Creating the Feature Dataset

In [ ]:
def create_feature_dataset(
    paths_subset_df, 
    links_map, 
    embedding_map, 
    dist_matrix, 
    article_id_to_index, 
    pagerank_map, 
    out_degree_map, 
    link_position_map,
    extractor
):
    X_features = []
    y_labels = []
    query_groups = []
    
    for row in tqdm(paths_subset_df.itertuples(), total=len(paths_subset_df), desc="Processing Paths"):
        path_articles = row.path.split(';')
        goal_id = path_articles[-1]
        
        for i in range(len(path_articles) - 1):
            source_id = path_articles[i]
            positive_target_id = path_articles[i+1]

            query_id = f"{row.Index}_{i}"

            all_candidates = links_map.get(source_id, [])
            if not all_candidates:
                continue 

            current_page_positions = link_position_map.get(source_id, {})

            emb_source = embedding_map.get(source_id, np.zeros(384))
            emb_goal = embedding_map.get(goal_id, np.zeros(384))
            source_idx = article_id_to_index.get(source_id, -1)
            goal_idx = article_id_to_index.get(goal_id, -1)
            
            if source_idx == -1 or goal_idx == -1:
                continue
                
            for candidate_id in all_candidates:
                if candidate_id not in article_id_to_index:
                    continue
                is_positive = 1 if (candidate_id == positive_target_id) else 0
                emb_candidate = embedding_map.get(candidate_id, np.zeros(384))
                candidate_idx = article_id_to_index.get(candidate_id, -1)
                sem_features = extractor.get_semantic_features(emb_source, emb_candidate, emb_goal)
                path_features = extractor.get_shortest_path_features(source_idx, candidate_idx, goal_idx, dist_matrix)
                topo_features = extractor.get_topology_features(candidate_id, pagerank_map, out_degree_map)
                pos_feature = current_page_positions.get(candidate_id, 1.0)

                all_features = {
                    **sem_features, 
                    **path_features, 
                    **topo_features,
                    "link_position": pos_feature 
                }
                
                X_features.append(all_features)
                y_labels.append(is_positive)
                query_groups.append(query_id)

    X_df = pd.DataFrame(X_features)
    y_series = pd.Series(y_labels, name="is_positive")
    query_series = pd.Series(query_groups, name="query_id")
    
    return X_df, y_series, query_series

## Splitting Data and Running the Pipeline

In [ ]:
train_paths_df, val_paths_df = train_test_split(
    paths_finished_df, 
    test_size=0.2, 
    random_state=42
)

print(f"Training paths: {len(train_paths_df)}")
print(f"Validation paths: {len(val_paths_df)}")

print("\nGenerating training data...")
X_train_df, y_train, q_train = create_feature_dataset(
    train_paths_df, links_map, embedding_map, dist_matrix, 
    article_id_to_index, pagerank_map, out_degree_map, 
    link_position_map,
    extractor
)

print("\nGenerating validation data...")
X_val_df, y_val, q_val = create_feature_dataset(
    val_paths_df, links_map, embedding_map, dist_matrix, 
    article_id_to_index, pagerank_map, out_degree_map, 
    link_position_map,
    extractor
)

print("\n--- Feature Dataset Shapes ---")
print(f"X_train: {X_train_df.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val_df.shape}, y_val: {y_val.shape}")

print("\n--- Example Features ---")
display(X_train_df.head())

print("\n--- Target Distribution (Train) ---")
print(y_train.value_counts(normalize=True))

## Logistic regression (P2 Milestone PoC) - model training

In [ ]:
X_train_df.replace([np.inf, -np.inf], 1000, inplace=True)
X_val_df.replace([np.inf, -np.inf], 1000, inplace=True)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_df)
X_val_scaled = scaler.transform(X_val_df)

print("Training Proof of Concept model (Logistic Regression)...")
model_poc = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model_poc.fit(X_train_scaled, y_train)

feature_importance = pd.DataFrame({
    'feature': X_train_df.columns,
    'coef': model_poc.coef_[0]
}).sort_values('coef', ascending=False)

print("\n--- Feature Importance (Model Coefficients) ---")
display(feature_importance)

## Logistic regression (P2 Milestone PoC) - evaluation

In [ ]:
print("Calculating Mean Reciprocal Rank (MRR) on validation set...")

mrr_score = calculate_mrr(
    model=model_poc,
    X_val=X_val_scaled,
    y_val=y_val,
    query_groups=q_val
)

print("\n" + "="*30)
print(f"   Mean Reciprocal Rank (MRR): {mrr_score:.4f}")
print("="*30)

# Heuristic Approach
Applying the heuristic that combines semantic similarity and hub-seeking behavior

$$
\text{Score}(c)
=
\alpha \cdot \cos(\mathbf{v}_c, \mathbf{v}_g)
\;+\;
(1 - \alpha) \cdot 
\frac{\text{outdeg}(c)}{\max_{c'} \text{outdeg}(c')}
$$

- c = candidate next article  
- g = goal article  
- $v_c$ = embedding vector of the candidate article  
- $v_g$ = embedding vector of the goal article  
- $cos(v_c, v_g)$ = cosine similarity between candidate and goal  
- $outdeg(c)$ = number of outgoing links of candidate (hub measure)  
- $max_{c'} {outdeg}(c')$ = maximum outdegree among all candidates (used to normalize)  
- $\alpha$ in [0,1] = controls trade-off between semantic similarity and hub-seeking  
  - Early in the game → larger $\alpha$ for hub-seeking  
  - Later in the game → smaller $\alpha$ to emphasize semantic closeness

In [ ]:
# formatted paths_finished df to apply the heuristic to

formatted_paths_finished_df = expand_paths_to_steps(paths_finished_df)

display(formatted_paths_finished_df)

In [ ]:
# Loop through formatted_paths_finished_df and calculate heuristic

selected_ids = set()
ranks = []

for _ in range(1000):

    # randomly choose an index that is unique
    while True:
        i = random.randint(0, len(formatted_paths_finished_df) - 1)
        if i not in selected_ids:
            selected_ids.add(i)
            break

    row = formatted_paths_finished_df.iloc[i]
    current_article_id = row["current_article"]
    goal_article_id = row["goal_article"]
    next_article_id = row["next_article"]
    path_length = row["path_length"]
    step_number = row["step_number"]
    alpha = step_number / path_length

    rank = compute_mrr_next_link(current_article_id, goal_article_id, next_article_id, links_map, embedding_map, articles_df, alpha=alpha)

    ranks.append(rank)

mrr = sum(ranks) / len(ranks)
print(f"Mean Reciprocal Rank (MRR): {mrr:.4f}")


# Graph Neural Network Approach
Applying a Graph Neural Network (GraphSAGE) to learn contextualized node embeddings that capture both semantic content and local graph topology.

$$
\mathbf{h}_i = \sigma \left( \mathbf{W} \cdot \text{AGG} \left( \{ \mathbf{x}_i \} \cup \{ \mathbf{x}_j, \forall j \in \mathcal{N}(i) \} \right) \right)
$$

$$
\text{Score}(c) = \text{MLP} \left( \mathbf{h}_s \;\|\; \mathbf{h}_c \;\|\; \mathbf{h}_g \right)
$$

- $s, c, g$ = source (current), candidate, and goal articles
- $\mathbf{x}_i$ = initial SBERT embedding of article $i$ (semantic features)
- $\mathbf{h}_i$ = learned GNN embedding of article $i$ (contextualized features)
- $\mathcal{N}(i)$ = set of immediate neighbors of article $i$ in the graph
- $\text{AGG}$ = aggregation function (e.g., mean) combining neighbor information
- $\|$ = concatenation operation
- $\text{MLP}$ = Multi-Layer Perceptron (neural network) that outputs the likelihood of clicking the link
- **Intuition**: Unlike the heuristic, the GNN learns the optimal non-linear combination of semantic similarity ($\mathbf{x}$) and graph structure ($\mathcal{N}$) directly from historical human paths.

### Training Loop

In [ ]:
# Device usage
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Running on Apple MPS")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("Running on CUDA")
else:
    DEVICE = torch.device("cpu")
    print("Running on cpu")

# Learning parameters
BATCH_SIZE = 2048
NUM_EPOCHS = 10
LR = 0.001
NEG_SAMPLES = 5

print("--- PREPARING GNN DATA ---")
pyg_data, article_id_to_idx = create_pyg_graph(articles_df, links_df, embedding_map)
pyg_data = pyg_data.to(DEVICE)

train_dataset = prepare_all_training_data(train_paths_df, article_id_to_idx, links_map, dist_matrix, neg_samples_ratio=NEG_SAMPLES)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

def train_and_analyze(model_type, epochs=10):
    print(f"\n{'='*40}")
    print(f"TRAINING MODEL: {model_type}")
    print(f"{'='*40}")
    
    model = NavigationGNN(in_channels=384, hidden_channels=128, model_type=model_type).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = torch.nn.BCEWithLogitsLoss()
    
    history = {'loss': [], 'mrr': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            src, cand, goal, dists, labels = batch
            src, cand, goal, dists, labels = src.to(DEVICE), cand.to(DEVICE), goal.to(DEVICE), dists.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            scores = model(pyg_data.x, pyg_data.edge_index, src, cand, goal, dists).squeeze()
            loss = criterion(scores, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        avg_loss = total_loss / len(train_loader)
        history['loss'].append(avg_loss)
        
        val_mrr = evaluate_gnn_mrr(model, pyg_data, val_paths_df, article_id_to_idx, links_map, dist_matrix, DEVICE, sample_size=500)
        history['mrr'].append(val_mrr)
        print(f"Epoch {epoch+1}: Loss {avg_loss:.4f} | Val MRR {val_mrr:.4f}")
        
    print(f"\n🔍 Running Detailed Error Analysis for {model_type}...")
    analysis_df = run_detailed_error_analysis(model, pyg_data, val_paths_df, article_id_to_idx, links_map, dist_matrix, DEVICE)
    plot_error_analysis(analysis_df, title_prefix=f"Model {model_type}")
    return history, model, analysis_df['mrr'].mean()

### Plots

In [ ]:
print("\n--- EXPERIMENT 1: GraphSAGE ---")
hist_sage, model_sage, final_mrr_sage = train_and_analyze("SAGE", epochs=NUM_EPOCHS)

print("\n--- EXPERIMENT 2: GATv2 ---")
hist_gat, model_gat, final_mrr_gat = train_and_analyze("GAT", epochs=NUM_EPOCHS)

# Final Comparison

In [ ]:
print("\n-- FINAL COMPARISON --")
plt.figure(figsize=(12, 6))
     
lr_mrr = 0.4736
heuristic_mrr = 0.1739 

plt.axhline(y=lr_mrr, color='gray', linestyle='--', label=f'Logistic Regression (PoC): {lr_mrr:.3f}')
plt.axhline(y=heuristic_mrr, color='green', linestyle=':', label=f'Heuristic: {heuristic_mrr:.3f}')

plt.plot(hist_sage['mrr'], label=f'Hybrid GraphSAGE (Best: {max(hist_sage["mrr"]):.3f})', marker='o', linewidth=2)
plt.plot(hist_gat['mrr'], label=f'Hybrid GATv2 (Best: {max(hist_gat["mrr"]):.3f})', marker='s', linewidth=2)

plt.title('Model Comparison: MRR over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Validation MRR')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()